# AutoMode — Gemma-2-2B-IT GSM8K Final Test (NeurIPS)
**All fixes applied:** prompt masking, device_map, LoRA→FFT merge, FFT→LoRA reset, lm_head freeze, step-then-switch ordering, scheduler continuity.

## Cell 1 — Install dependencies

In [ ]:
!pip install -q -U transformers datasets accelerate peft evaluate sentencepiece bitsandbytes scikit-learn pandas numpy tqdm trl huggingface_hub
# LoRA-GA custom PEFT fork — install from local repo
import sys
!{sys.executable} -m pip uninstall -y peft
!cd /workspace/LoRA-GA && {sys.executable} -m pip install -e peft --no-deps

## Cell 2 — HuggingFace Login

In [ ]:
import os
from huggingface_hub import login
# Read token from environment variable — never hardcode tokens in notebooks
hf_token = os.environ.get("HF_TOKEN", "")
if not hf_token:
    raise ValueError("Set the HF_TOKEN environment variable before running. e.g. export HF_TOKEN=hf_...")
login(token=hf_token)
print("HuggingFace login OK")

HuggingFace login OK


## Cell 3 — Imports

In [65]:
import os, re, gc, copy, json, math, time, random, hashlib, inspect
from collections import Counter
from collections.abc import Mapping
from dataclasses import dataclass, asdict, field
from itertools import islice
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from datasets import load_dataset, Dataset, DatasetDict
from datasets.utils.logging import disable_progress_bar
disable_progress_bar()

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
)
from peft import (
    LoraConfig,
    AdaLoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
print("Imports OK")

Imports OK


## Cell 4 — PEFT Feature Detection

In [66]:
import peft
print("peft version:", getattr(peft, '__version__', 'unknown'))
print("peft file:", peft.__file__)

PEFT_SUPPORTS_DORA   = False
PEFT_SUPPORTS_LORAGA = False
LoraGAConfig = estimate_gradient = LoraGAContext = None
save_loraga_model_init = save_loraga_model_final = None

try:
    PEFT_SUPPORTS_DORA = "use_dora" in inspect.signature(LoraConfig.__init__).parameters
except Exception:
    pass

try:
    from peft import LoraGAConfig as _LGC, get_peft_model
    from peft.utils.lora_ga_utils import (
        estimate_gradient as _eg,
        LoraGAContext as _LC,
        save_loraga_model_init as _si,
        save_loraga_model_final as _sf,
    )
    LoraGAConfig = _LGC
    estimate_gradient = _eg
    LoraGAContext = _LC
    save_loraga_model_init = _si
    save_loraga_model_final = _sf
    PEFT_SUPPORTS_LORAGA = True
except Exception as e:
    print("LoRA-GA repo import failed:", repr(e))

print({"PEFT_SUPPORTS_DORA": PEFT_SUPPORTS_DORA, "PEFT_SUPPORTS_LORAGA": PEFT_SUPPORTS_LORAGA})

peft version: 0.11.2.dev0
peft file: /workspace/LoRA-GA/peft/src/peft/__init__.py
{'PEFT_SUPPORTS_DORA': True, 'PEFT_SUPPORTS_LORAGA': True}


## Cell 5 — Output Folder Setup

In [67]:
import shutil

FINAL_ROOT = "runs_gemma2_2b_gsm8k_final_test"

root = Path(FINAL_ROOT)
root.mkdir(parents=True, exist_ok=True)
existing = [x for x in root.iterdir()] if root.exists() else []
if existing:
    print(f"Resuming into existing folder: {root} ({len(existing)} items found)")
else:
    print(f"Fresh final-test folder ready: {root}")

Fresh final-test folder ready: runs_gemma2_2b_gsm8k_final_test


## Cell 6 — RunConfig

In [68]:
@dataclass
class RunConfig:
    model_name:              str   = "google/gemma-2-2b-it"
    use_4bit:                bool  = False
    dtype_str:               str   = "bfloat16"
    gradient_checkpointing:  bool  = True

    train_track:   str           = "gsm8k"
    mode:          str           = "smoke"   # smoke | screen | final_test
    eval_buckets:  Optional[List[str]] = None

    method: str = "lora"  # full_ft | lora | bitfit | topk_full | automode | adalora | dora | loraga

    seed:          int   = 8
    epochs:        int   = 2
    learning_rate: float = 5e-5
    weight_decay:  float = 0.01
    warmup_ratio:  float = 0.03

    train_batch_size: int   = 1
    grad_accum_steps: int   = 16
    max_grad_norm:    float = 1.0

    max_train_samples: Optional[int] = None
    max_eval_samples:  Optional[int] = None

    max_source_len: int   = 384
    max_target_len: int   = 128
    max_new_tokens: int   = 256
    num_beams:      int   = 1
    do_sample:      bool  = True
    temperature:    float = 0.7
    top_p:          float = 0.95
    sampling_k:     int   = 3
    eval_batch_size: int = 8

    lora_r:              int            = 16
    lora_alpha:          int            = 32
    lora_dropout:        float          = 0.05
    lora_target_modules: Tuple[str,...] = ("q_proj", "v_proj")

    adalora_init_r:   int = 16
    adalora_target_r: int = 8
    adalora_tinit:    int = 200
    adalora_tfinal:   int = 1000
    adalora_deltaT:   int = 10

    topk_k:          int = 6
    topk_strategy:   str = "last_k"   # last_k | deep_block
    deep_block_start: int = 13
    deep_block_end:   int = 19

    dynamic_updates:   int = 10
    dynamic_threshold: int = 10

    loraga_steps_for_grad_est: int = 16

    output_root: str  = FINAL_ROOT
    save_model:  bool = False   # set True only when you need the checkpoint

    mtbench_question_path_env: str = "MTBENCH_QUESTIONS_PATH"

    gsm8k_eval_split:        str   = "test"   # always test for final NeurIPS runs
    gsm8k_val_ratio:         float = 0.1
    gsm8k_split_seed:        int   = 123
    gsm8k_train_on_full_train: bool = True    # True for final runs

cfg = RunConfig()
print(cfg)

RunConfig(model_name='google/gemma-2-2b-it', use_4bit=False, dtype_str='bfloat16', gradient_checkpointing=True, train_track='gsm8k', mode='smoke', eval_buckets=None, method='lora', seed=8, epochs=2, learning_rate=5e-05, weight_decay=0.01, warmup_ratio=0.03, train_batch_size=1, grad_accum_steps=16, max_grad_norm=1.0, max_train_samples=None, max_eval_samples=None, max_source_len=384, max_target_len=128, max_new_tokens=256, num_beams=1, do_sample=True, temperature=0.7, top_p=0.95, sampling_k=3, eval_batch_size=8, lora_r=16, lora_alpha=32, lora_dropout=0.05, lora_target_modules=('q_proj', 'v_proj'), adalora_init_r=16, adalora_target_r=8, adalora_tinit=200, adalora_tfinal=1000, adalora_deltaT=10, topk_k=6, topk_strategy='last_k', deep_block_start=13, deep_block_end=19, dynamic_updates=10, dynamic_threshold=10, loraga_steps_for_grad_est=16, output_root='runs_gemma2_2b_gsm8k_final_test', save_model=False, mtbench_question_path_env='MTBENCH_QUESTIONS_PATH', gsm8k_eval_split='test', gsm8k_val_r

## Cell 7 — Utility Functions

In [69]:
DTYPE_MAP = {
    "float16":  torch.float16,
    "bfloat16": torch.bfloat16,
    "float32":  torch.float32,
}

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def make_run_id(cfg: RunConfig) -> str:
    payload = {
        "model": cfg.model_name, "track": cfg.train_track, "mode": cfg.mode,
        "method": cfg.method, "seed": cfg.seed, "lr": cfg.learning_rate,
        "epochs": cfg.epochs, "lora_r": cfg.lora_r, "lora_alpha": cfg.lora_alpha,
        "dynamic_updates": cfg.dynamic_updates, "dynamic_threshold": cfg.dynamic_threshold,
        "topk_k": cfg.topk_k, "topk_strategy": cfg.topk_strategy,
    }
    return hashlib.md5(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:12]

def ensure_dirs(cfg: RunConfig, run_id: str):
    root    = Path(cfg.output_root)
    run_dir = root / run_id
    paths   = {
        "root": root, "run": run_dir,
        "configs": run_dir / "configs", "logs": run_dir / "logs",
        "evals":   run_dir / "evals",   "dynamic": run_dir / "dynamic",
        "checkpoints": run_dir / "checkpoints", "preds": run_dir / "preds",
    }
    for p in paths.values():
        p.mkdir(parents=True, exist_ok=True)
    return paths

def save_json(obj: Dict[str, Any], path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def method_trains_base_weights(method: str) -> bool:
    return method in {"full_ft", "bitfit", "topk_full", "automode"}

def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    return trainable, total

def get_layer_index_from_name(name: str) -> Optional[int]:
    m = re.search(r"model\.layers\.(\d+)", name)
    return int(m.group(1)) if m else None

def get_all_transformer_layer_ids(model) -> List[int]:
    ids = set()
    for n, _ in model.named_parameters():
        idx = get_layer_index_from_name(n)
        if idx is not None:
            ids.add(idx)
    return sorted(ids)

def majority_vote(items):
    items = [x for x in items if x is not None and str(x).strip() != ""]
    if not items:
        return None
    return Counter(items).most_common(1)[0][0]

print("Utility functions defined.")

Utility functions defined.


## Cell 8 — Dataset Loaders

In [70]:
def load_gsm8k_dataset() -> DatasetDict:
    return load_dataset("openai/gsm8k", "main")

def load_math_dataset() -> DatasetDict:
    candidates = [
        ("DigitalLearningGmbH/MATH-lighteval", None),
        ("lighteval/MATH", None),
        ("hendrycks/competition_math", None),
    ]
    for name, subset in candidates:
        try:
            return load_dataset(name) if subset is None else load_dataset(name, subset)
        except Exception:
            pass
    raise RuntimeError("Could not load any MATH dataset candidate.")

def load_mmlu_dataset() -> DatasetDict:
    return load_dataset("cais/mmlu", "all")

def load_arc_dataset() -> DatasetDict:
    return load_dataset("allenai/ai2_arc", "ARC-Challenge")

def normalize_gsm8k_dataset(ds: Dataset) -> Dataset:
    """Converts raw GSM8K split into unified schema with prompt/target/raw_question/raw_answer."""
    def _map(ex):
        q = ex["question"].strip()
        a = ex["answer"].strip()
        return {
            "prompt": (
                "Solve the following math word problem step by step. "
                "End your response with the final answer in the format '#### <number>'.\n\n"
                f"Question: {q}\n\nAnswer:"
            ),
            "target":       a,
            "raw_question": q,
            "raw_answer":   a,
        }
    keep = ["prompt", "target", "raw_question", "raw_answer"]
    ds   = ds.map(_map)
    drop = [c for c in ds.column_names if c not in keep]
    return ds.remove_columns(drop) if drop else ds

def get_gsm8k_screen_splits(cfg: RunConfig) -> Dict:
    ds         = load_gsm8k_dataset()
    train_full = ds["train"]
    test_split = ds["test"]
    if cfg.gsm8k_train_on_full_train:
        train_split = train_full
        val_split   = None
    else:
        sp          = train_full.train_test_split(
            test_size=cfg.gsm8k_val_ratio, seed=cfg.gsm8k_split_seed, shuffle=True
        )
        train_split = sp["train"]
        val_split   = sp["test"]
    return {"train": train_split, "val": val_split, "test": test_split}

def load_training_track(cfg: RunConfig, tokenizer) -> DatasetDict:
    if cfg.train_track == "gsm8k":
        splits   = get_gsm8k_screen_splits(cfg)
        train_ds = normalize_gsm8k_dataset(splits["train"])
        return DatasetDict({"train": train_ds})
    raise ValueError(f"Unknown train_track: {cfg.train_track}")

print("Dataset loaders defined.")

Dataset loaders defined.


## Cell 9 — Tokenizer, Prompt Builders, Data Pipeline

In [71]:
def get_tokenizer(model_name: str):
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    return tok

def build_mmlu_prompt(question, choices, fewshot_examples):
    letters = ["A", "B", "C", "D"]
    header  = "You are taking a multiple-choice exam. Reply with only the correct option letter.\n\n"
    shots   = []
    for ex in fewshot_examples:
        shot = ex["question"] + "\n"
        for i, c in enumerate(ex["choices"]):
            shot += f"{letters[i]}. {c}\n"
        shot += f"Answer: {letters[ex['answer']]}\n\n"
        shots.append(shot)
    cur = question + "\n"
    for i, c in enumerate(choices):
        cur += f"{letters[i]}. {c}\n"
    cur += "Answer:"
    return header + "".join(shots) + cur

def build_arc_prompt(question, choices):
    letters = ["A", "B", "C", "D", "E"]
    prompt  = "Answer the science multiple-choice question. Reply with only the correct option letter.\n\n"
    prompt += question + "\n"
    for i, c in enumerate(choices):
        prompt += f"{letters[i]}. {c}\n"
    return prompt + "Answer:"

# ── FIXED: prompt masking ────────────────────────────────────────────────────
# Original code set labels = input_ids for ALL tokens, computing loss over the
# prompt. We now mask prompt tokens with -100 so loss is only on answer tokens.
def tokenize_sft_batch(batch, tokenizer, max_source_len=384, max_target_len=128):
    prompts = batch["prompt"]
    targets = batch["target"]

    # Tokenize prompts alone to get exact lengths
    prompt_only = tokenizer(
        prompts,
        max_length=max_source_len,
        truncation=True,
        padding=False,
        add_special_tokens=True,
    )

    # Tokenize full sequences
    full_texts = [p + " " + t for p, t in zip(prompts, targets)]
    tokenized  = tokenizer(
        full_texts,
        max_length=max_source_len + max_target_len,
        truncation=True,
        padding=False,
        add_special_tokens=True,
    )

    labels = []
    for i, input_ids in enumerate(tokenized["input_ids"]):
        prompt_len = len(prompt_only["input_ids"][i])
        lbl        = input_ids.copy()
        lbl[:prompt_len] = [-100] * prompt_len   # mask prompt tokens
        labels.append(lbl)

    tokenized["labels"] = labels
    return tokenized

def build_train_dataset(ds: Dataset, tokenizer, cfg: RunConfig):
    if cfg.max_train_samples is not None:
        ds = ds.select(range(min(cfg.max_train_samples, len(ds))))
    return ds.map(
        lambda x: tokenize_sft_batch(x, tokenizer, cfg.max_source_len, cfg.max_target_len),
        batched=True,
        remove_columns=ds.column_names,
    )

class CausalLMCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
    def __call__(self, features):
        return self.tokenizer.pad(features, padding=True, return_tensors="pt")

def default_eval_buckets(train_track: str, mode: str = "smoke") -> List[str]:
    if train_track == "gsm8k":
        return ["gsm8k"]
    raise ValueError(f"Unknown train_track={train_track}")

print("Tokenizer and data pipeline defined.")

Tokenizer and data pipeline defined.


## Cell 10 — Model Loading

In [72]:
# ── FIXED: device_map ────────────────────────────────────────────────────────
# Original: device_map="auto" can shard model across GPUs causing device
# mismatch errors during backward when AutoMode moves requires_grad around.
# Fixed: device_map={"":"cuda:0"} pins everything to a single GPU explicitly.
def load_base_model(cfg: RunConfig):
    dtype = DTYPE_MAP[cfg.dtype_str]

    if cfg.method == "loraga":
        use_4bit   = False
        device_map = None
    else:
        use_4bit   = cfg.use_4bit and (not method_trains_base_weights(cfg.method))
        device_map = {"" : "cuda:0"} if torch.cuda.is_available() else None

    model_kwargs = {"device_map": device_map}

    try:
        model_kwargs["dtype"] = dtype
        if use_4bit:
            from transformers import BitsAndBytesConfig
            model_kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=dtype,
            )
        model = AutoModelForCausalLM.from_pretrained(cfg.model_name, **model_kwargs)
    except TypeError:
        model_kwargs.pop("dtype", None)
        model_kwargs["torch_dtype"] = dtype
        model = AutoModelForCausalLM.from_pretrained(cfg.model_name, **model_kwargs)

    if cfg.method == "loraga" and torch.cuda.is_available():
        model = model.to("cuda:0")

    if cfg.gradient_checkpointing:
        model.gradient_checkpointing_enable()
        model.config.use_cache = False

    if use_4bit:
        try:
            model = prepare_model_for_kbit_training(model)
        except Exception:
            pass

    return model

print("load_base_model defined.")

load_base_model defined.


## Cell 11 — Method-Specific Model Builders (LoRA, Full FT, BitFit, TopK, AdaLoRA, DoRA, LoRA-GA)

In [73]:
def apply_full_ft(model):
    for p in model.parameters():
        p.requires_grad = True
    return model

def apply_lora(model, cfg: RunConfig):
    peft_cfg = LoraConfig(
        r=cfg.lora_r, lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=list(cfg.lora_target_modules),
        bias="none", task_type="CAUSAL_LM",
    )
    return get_peft_model(model, peft_cfg)

def apply_dora(model, cfg: RunConfig):
    if not PEFT_SUPPORTS_DORA:
        raise RuntimeError("Installed PEFT does not support DoRA.")
    peft_cfg = LoraConfig(
        r=cfg.lora_r, lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=list(cfg.lora_target_modules),
        bias="none", task_type="CAUSAL_LM", use_dora=True,
    )
    return get_peft_model(model, peft_cfg)

def _resolve_adalora_schedule(cfg: RunConfig, total_steps: int):
    if total_steps < 3:
        raise ValueError(f"AdaLoRA needs >=3 optimizer steps, got {total_steps}.")
    tinit    = min(cfg.adalora_tinit,  max(1, total_steps // 4))
    tfinal   = min(cfg.adalora_tfinal, max(1, total_steps - tinit - 1))
    deltaT   = min(cfg.adalora_deltaT, max(1, total_steps - tinit - tfinal))
    return tinit, tfinal, deltaT

def apply_adalora(model, cfg: RunConfig, total_steps: int):
    tinit, tfinal, deltaT = _resolve_adalora_schedule(cfg, total_steps)
    print(f"[adalora] total_steps={total_steps}, tinit={tinit}, tfinal={tfinal}, deltaT={deltaT}")
    peft_cfg = AdaLoraConfig(
        init_r=cfg.adalora_init_r, target_r=cfg.adalora_target_r,
        lora_alpha=cfg.lora_alpha, lora_dropout=cfg.lora_dropout,
        target_modules=list(cfg.lora_target_modules),
        task_type="CAUSAL_LM", bias="none",
        total_step=total_steps, tinit=tinit, tfinal=tfinal, deltaT=deltaT,
    )
    return get_peft_model(model, peft_cfg)

def apply_bitfit(model):
    for p in model.parameters():
        p.requires_grad = False
    enabled = []
    for name, p in model.named_parameters():
        if name.endswith("bias") or ".bias" in name:
            p.requires_grad = True
            enabled.append(name)
    if not enabled:
        print("[bitfit] No bias tensors. Falling back to 1D parameters.")
        for name, p in model.named_parameters():
            if p.ndim == 1 and "lora_" not in name:
                p.requires_grad = True
                enabled.append(name)
    if not enabled:
        print("[bitfit] No 1D params. Falling back to lm_head.")
        for name, p in model.named_parameters():
            if "lm_head" in name:
                p.requires_grad = True
                enabled.append(name)
    if not enabled:
        raise RuntimeError("BitFit: no trainable parameters found.")
    print(f"[bitfit] enabled_tensors={len(enabled)}, sample={enabled[:5]}")
    return model

def apply_topk_full_ft(model, cfg: RunConfig):
    layer_ids = get_all_transformer_layer_ids(model)
    if cfg.topk_strategy == "last_k":
        selected = set(layer_ids[-cfg.topk_k:])
    elif cfg.topk_strategy == "deep_block":
        selected = {i for i in layer_ids if cfg.deep_block_start <= i <= cfg.deep_block_end}
    else:
        raise ValueError(f"Unknown topk_strategy: {cfg.topk_strategy}")
    for name, p in model.named_parameters():
        idx = get_layer_index_from_name(name)
        p.requires_grad = (idx in selected)
    for name, p in model.named_parameters():
        if "lm_head" in name:
            p.requires_grad = True
    print(f"[topk] selected layers: {sorted(selected)}")
    return model

def _move_batch_to_device(batch, device):
    if hasattr(batch, "to"):
        try: return batch.to(device)
        except Exception: pass
    if torch.is_tensor(batch):
        return batch.to(device, non_blocking=True)
    if isinstance(batch, Mapping):
        return {k: _move_batch_to_device(v, device) for k, v in batch.items()}
    if isinstance(batch, (list, tuple)):
        return type(batch)(_move_batch_to_device(x, device) for x in batch)
    return batch

class DeviceLoader:
    def __init__(self, loader, device):
        self.loader = loader
        self.device = device
    def __iter__(self):
        for batch in self.loader:
            yield _move_batch_to_device(batch, self.device)
    def __len__(self):
        return len(self.loader)

def apply_loraga(model, cfg: RunConfig, grad_estimation_loader: DataLoader):
    if not PEFT_SUPPORTS_LORAGA:
        raise RuntimeError("LoRA-GA repo PEFT not available.")
    if cfg.use_4bit:
        raise ValueError("LoRA-GA requires floating-point model. Set use_4bit=False.")
    if not torch.cuda.is_available():
        raise RuntimeError("LoRA-GA requires CUDA.")
    device     = torch.device("cuda:0")
    model      = model.to(device)
    model.train()
    max_batches = min(cfg.loraga_steps_for_grad_est, len(grad_estimation_loader))
    peft_config = LoraGAConfig(
        task_type="CAUSAL_LM", r=cfg.lora_r, lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout, target_modules=list(cfg.lora_target_modules),
        bias="none", bsz=cfg.train_batch_size, iters=max_batches,
        max_length=cfg.max_source_len + cfg.max_target_len,
        dtype="bf16" if cfg.dtype_str == "bfloat16" else ("fp16" if cfg.dtype_str == "float16" else "fp32"),
        direction="ArB2r", scale="stable", stable_gamma=16,
    )
    small_loader = list(islice(grad_estimation_loader, max_batches))
    grad_loader  = DeviceLoader(small_loader, device)
    named_grad   = estimate_gradient(model=model, dataloader=grad_loader, accelerator=None, quant_flag=False)
    with LoraGAContext(model=model, named_grad=named_grad):
        model = get_peft_model(model=model, peft_config=peft_config)
    return model

def build_loraga_grad_loader(train_tokenized, collator, cfg: RunConfig):
    n  = min(len(train_tokenized), cfg.loraga_steps_for_grad_est * cfg.train_batch_size)
    ds = train_tokenized.shuffle(seed=cfg.seed).select(range(n))
    return DataLoader(ds, batch_size=cfg.train_batch_size, shuffle=False, collate_fn=collator)

print("Method builders defined.")

Method builders defined.


## Cell 12 — AutoMode Core (merge/reset/switch/init/optimizers) — ALL FIXED

In [74]:
# ════════════════════════════════════════════════════════════════════
# AUTOMODE CORE — complete rewrite with all four fixes applied
# ════════════════════════════════════════════════════════════════════

def collect_layer_grad_scores(model) -> Dict[int, float]:
    """
    Computes S_l = sqrt( (1/|P_l|) * sum_p ||grad_p L||^2 ) for each layer.
    Must be called BEFORE optimizer.zero_grad() — reads .grad tensors.
    Only includes parameters that are currently trainable (requires_grad=True)
    and have a non-None gradient, so LoRA and FFT layers are compared fairly
    on their respective active parameter sets.
    """
    layer_sums   = {}
    layer_counts = {}
    for name, p in model.named_parameters():
        if p.grad is None or not p.requires_grad:
            continue
        idx = get_layer_index_from_name(name)
        if idx is None:
            continue
        g2  = float(torch.sum(p.grad.detach().float() ** 2).item())
        cnt = p.numel()
        layer_sums[idx]   = layer_sums.get(idx, 0.0) + g2
        layer_counts[idx] = layer_counts.get(idx, 0)  + cnt
    return {
        idx: math.sqrt(layer_sums[idx] / max(layer_counts[idx], 1))
        for idx in layer_sums
    }


def _merge_lora_into_base(model, layer_idx: int):
    """
    FIX — called when upgrading a layer from LoRA → FFT.

    Paper spec: θ_l ← θ_l + (α/r)·B_l·A_l, then deactivate adapters.

    Without this, the layer in 'FFT mode' is actually doing
    base_weight(x) + frozen_adapter(x), NOT pure FFT.
    PEFT's merge_adapter() fuses the adapter into the base weight and
    bypasses the adapter path in all subsequent forward passes.
    """
    prefix = f"model.layers.{layer_idx}."
    for mod_name, module in model.named_modules():
        if not mod_name.startswith(prefix):
            continue
        if not hasattr(module, "merge_adapter"):
            continue
        is_merged = getattr(module, "merged", False)
        if isinstance(is_merged, list):
            is_merged = len(is_merged) > 0
        if not is_merged:
            try:
                module.merge_adapter()
            except Exception as e:
                print(f"[automode] WARNING merge_adapter failed on {mod_name}: {e}")


def _reset_lora_adapters(model, layer_idx: int):
    """
    FIX — called when downgrading a layer from FFT → LoRA.

    Paper spec: freeze current weights, create a fresh initialized adapter pair.

    Implementation:
      1. Unmerge (separates fused base weight from adapter path).
      2. Zero lora_B so the adapter output starts at zero — functionally
         equivalent to a fresh adapter since BA=0 when B=0.
    """
    prefix = f"model.layers.{layer_idx}."
    for mod_name, module in model.named_modules():
        if not mod_name.startswith(prefix):
            continue
        if hasattr(module, "unmerge_adapter"):
            is_merged = getattr(module, "merged", False)
            if isinstance(is_merged, list):
                is_merged = len(is_merged) > 0
            if is_merged:
                try:
                    module.unmerge_adapter()
                except Exception as e:
                    print(f"[automode] WARNING unmerge_adapter failed on {mod_name}: {e}")
        for pname, p in module.named_parameters(recurse=False):
            if "lora_B" in pname:
                with torch.no_grad():
                    p.zero_()


_ADAPTER_TAGS = frozenset([
    "lora_A", "lora_B",
    "lora_embedding_A", "lora_embedding_B",
    "magnitude_vector",
])

def set_automode_layer_mode(
    model,
    layer_idx: int,
    mode: str,
    prev_mode: Optional[str] = None,
):
    """
    Switch layer layer_idx between 'lora' and 'full_ft'.

    Merge/reset only fires on real transitions (prev_mode != mode) to
    avoid redundant work when reassignment leaves a layer in the same mode.

    LoRA mode:  only adapter params (lora_A, lora_B ...) are trainable.
    FFT mode:   base weights are trainable; adapter params are frozen
                (and merged into base weight so forward pass is pure FFT).
    """
    assert mode in {"full_ft", "lora"}, f"Bad mode: {mode}"
    prefix = f"model.layers.{layer_idx}."

    if mode == "full_ft" and prev_mode != "full_ft":
        _merge_lora_into_base(model, layer_idx)

    if mode == "lora" and prev_mode == "full_ft":
        _reset_lora_adapters(model, layer_idx)

    for name, p in model.named_parameters():
        if not name.startswith(prefix):
            continue
        is_adapter = any(tag in name for tag in _ADAPTER_TAGS)
        p.requires_grad = (not is_adapter) if mode == "full_ft" else is_adapter


def initialize_automode(model, cfg: RunConfig):
    """
    Sets up the model for AutoMode:
    1. Inject LoRA adapters.
    2. Explicitly freeze lm_head and embed_tokens — some PEFT versions
       incorrectly leave these trainable, inflating param count from ~5M
       to ~600M and causing OOM on init.
    3. Start all transformer layers in LoRA mode.
    """
    model = apply_lora(model, cfg)

    # Hard freeze — belt-and-suspenders against PEFT version differences
    for name, p in model.named_parameters():
        if "lm_head" in name or "embed_tokens" in name:
            p.requires_grad = False

    # All layers start in LoRA mode (prev_mode=None → no merge/reset)
    for idx in get_all_transformer_layer_ids(model):
        set_automode_layer_mode(model, idx, "lora", prev_mode=None)

    return model


def rebuild_optimizer(model, cfg: RunConfig, total_steps: int):
    """Full optimizer + scheduler build. Called ONCE at start of training."""
    params = [p for p in model.parameters() if p.requires_grad]

    # Full FT on 32GB: fp32 AdamW needs ~21GB optimizer states alone.
    # 8-bit AdamW quantizes moments to int8, reducing that to ~5GB.
    if cfg.method == "full_ft":
        try:
            import bitsandbytes as bnb
            optimizer = bnb.optim.AdamW8bit(
                params,
                lr=cfg.learning_rate,
                weight_decay=cfg.weight_decay,
            )
            print("[full_ft] Using 8-bit AdamW — saves ~16GB on 32GB GPU.")
        except Exception as e:
            print(f"[full_ft] 8-bit AdamW unavailable ({e}), using standard AdamW. May OOM.")
            optimizer = torch.optim.AdamW(
                params, lr=cfg.learning_rate, weight_decay=cfg.weight_decay
            )
    else:
        optimizer = torch.optim.AdamW(
            params, lr=cfg.learning_rate, weight_decay=cfg.weight_decay
        )

    warmup_steps = int(total_steps * cfg.warmup_ratio)
    scheduler    = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )
    return optimizer, scheduler


def rebuild_optimizer_after_switch(model, cfg: RunConfig):
    """Optimizer-only rebuild after mode switch. Scheduler never rebuilt."""
    params    = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        params, lr=cfg.learning_rate, weight_decay=cfg.weight_decay
    )
    return optimizer


print("AutoMode core defined (all fixes applied).")

AutoMode core defined (all fixes applied).


## Cell 13 — build_model_for_method

In [75]:
def build_model_for_method(cfg: RunConfig, train_loader_for_loraga=None, total_steps: Optional[int] = None):
    model = load_base_model(cfg)

    if   cfg.method == "full_ft":   model = apply_full_ft(model)
    elif cfg.method == "lora":      model = apply_lora(model, cfg)
    elif cfg.method == "bitfit":    model = apply_bitfit(model)
    elif cfg.method == "topk_full": model = apply_topk_full_ft(model, cfg)
    elif cfg.method == "automode":  model = initialize_automode(model, cfg)
    elif cfg.method == "dora":      model = apply_dora(model, cfg)
    elif cfg.method == "adalora":
        if total_steps is None:
            raise ValueError("AdaLoRA requires total_steps.")
        model = apply_adalora(model, cfg, total_steps)
    elif cfg.method == "loraga":
        if train_loader_for_loraga is None:
            raise ValueError("LoRA-GA requires a gradient-estimation loader.")
        model = apply_loraga(model, cfg, train_loader_for_loraga)
    else:
        raise ValueError(f"Unknown method: {cfg.method}")

    trainable, total = count_trainable_params(model)
    print({
        "method": cfg.method,
        "trainable_params": trainable,
        "total_params": total,
        "trainable_pct": round(100 * trainable / total, 4),
    })
    return model

print("build_model_for_method defined.")

build_model_for_method defined.


## Cell 14 — Generation and Answer Extraction

In [76]:
def _ensure_left_padding(tokenizer):
    """
    Decoder-only batched generation requires left padding so that
    all sequences in the batch end with real tokens, not pad tokens.
    The original tokenizer is right-padded for training — we switch
    here only for evaluation generation.
    """
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

def _ensure_right_padding(tokenizer):
    """Restore right-padding for training after eval."""
    tokenizer.padding_side = "right"

def _build_generate_kwargs(cfg, tokenizer) -> dict:
    kwargs = {
        "max_new_tokens": cfg.max_new_tokens,
        "pad_token_id":   tokenizer.pad_token_id,
        "eos_token_id":   tokenizer.eos_token_id,
        "use_cache":      True,
    }
    do_sample = getattr(cfg, "do_sample", True)
    kwargs["do_sample"] = do_sample
    if do_sample:
        kwargs["temperature"] = getattr(cfg, "temperature", 0.7)
        kwargs["top_p"]       = getattr(cfg, "top_p", 0.95)
    else:
        kwargs["num_beams"] = getattr(cfg, "num_beams", 1)
    return kwargs

def generate_text(model, tokenizer, prompt: str, cfg) -> str:
    """Single-example generation — used outside eval (e.g. quick checks)."""
    _ensure_left_padding(tokenizer)
    enc = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=cfg.max_source_len,
    )
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.inference_mode():
        outputs = model.generate(**enc, **_build_generate_kwargs(cfg, tokenizer))
    input_len  = enc["input_ids"].shape[1]
    gen_tokens = outputs[0, input_len:]
    return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

def generate_text_batch(model, tokenizer, prompts: List[str], cfg) -> List[str]:
    """
    Batched generation.
    Left-padding ensures all sequences in the batch end at the same
    position, so slicing outputs[:, input_len:] correctly strips
    the prompt from every row simultaneously.
    """
    _ensure_left_padding(tokenizer)
    enc = tokenizer(
        list(prompts), return_tensors="pt",
        padding=True, truncation=True, max_length=cfg.max_source_len,
    )
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.inference_mode():
        outputs = model.generate(**enc, **_build_generate_kwargs(cfg, tokenizer))
    input_len  = enc["input_ids"].shape[1]
    gen_tokens = outputs[:, input_len:]
    return [t.strip() for t in tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)]

def extract_hash_answer(text: Optional[str]) -> Optional[str]:
    if text is None:
        return None
    m = re.findall(r"####\s*([^\n]+)", text)
    return m[-1].strip() if m else None

def extract_boxed_answer(text: Optional[str]) -> Optional[str]:
    if text is None:
        return None
    m = re.findall(r"\\boxed\{([^}]*)\}", text)
    return m[-1].strip() if m else None

def normalize_answer_string(ans: Optional[str]) -> str:
    if ans is None:
        return ""
    return ans.strip().replace("$", "").replace(",", "").replace(" ", "").lower()

def extract_option_letter(text: str, allowed="ABCDE") -> Optional[str]:
    m = re.findall(rf"\b([{allowed}])\b", text.strip())
    return m[-1] if m else None

print("Generation utilities (batched) defined.")

Generation utilities (batched) defined.


## Cell 15 — Evaluation Functions

In [77]:
def evaluate_gsm8k(model, tokenizer, cfg: RunConfig, paths) -> Dict[str, Any]:
    """
    Batched GSM8K evaluation with num_return_sequences for speed.
    Scientifically identical to sampling_k separate calls:
      - same 1319 test examples
      - same sampling_k generations per example (done in 1 generate call)
      - same answer extraction, majority vote, maj@1 reporting
    """
    model.config.use_cache = True
    model.eval()

    splits = get_gsm8k_screen_splits(cfg)
    if cfg.gsm8k_eval_split == "val":
        ds = splits["val"]
        if ds is None:
            raise ValueError("GSM8K val split requested but none exists "
                             "(gsm8k_train_on_full_train=True).")
    elif cfg.gsm8k_eval_split == "test":
        ds = splits["test"]
    else:
        raise ValueError(f"Unknown gsm8k_eval_split: {cfg.gsm8k_eval_split}")

    ds = normalize_gsm8k_dataset(ds)
    if cfg.max_eval_samples is not None:
        ds = ds.select(range(min(cfg.max_eval_samples, len(ds))))

    eval_bs = cfg.eval_batch_size
    loader  = DataLoader(ds, batch_size=eval_bs, shuffle=False)

    local_cfg             = copy.deepcopy(cfg)
    local_cfg.do_sample   = True
    local_cfg.temperature = 0.7
    local_cfg.top_p       = 0.95

    k = max(1, cfg.sampling_k)

    correct = 0
    records = []

    for batch in tqdm(loader, desc=f"GSM8K {cfg.gsm8k_eval_split} eval", leave=False):
        prompts       = list(batch["prompt"])
        raw_questions = list(batch["raw_question"])
        raw_answers   = list(batch["raw_answer"])
        batch_size    = len(prompts)

        # Single generate call producing k samples per example.
        # output shape: (batch_size * k, seq_len)
        _ensure_left_padding(tokenizer)
        enc = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=cfg.max_source_len,
        )
        enc = {key: val.to(model.device) for key, val in enc.items()}

        gen_kwargs = _build_generate_kwargs(local_cfg, tokenizer)
        gen_kwargs["num_return_sequences"] = k

        with torch.inference_mode():
            outputs = model.generate(**enc, **gen_kwargs)

        input_len  = enc["input_ids"].shape[1]
        gen_tokens = outputs[:, input_len:]
        all_texts  = tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)
        # all_texts is flat: [ex0_sample0, ex0_sample1, ex0_sample2,
        #                      ex1_sample0, ex1_sample1, ex1_sample2, ...]

        # Collect votes per example
        all_votes: List[List[str]] = [[] for _ in range(batch_size)]
        for i in range(batch_size):
            for text in all_texts[i * k : (i + 1) * k]:
                pred = extract_hash_answer(text.strip())
                if pred is not None:
                    all_votes[i].append(pred)

        for q, gold_raw, votes in zip(raw_questions, raw_answers, all_votes):
            pred_final = majority_vote(votes) if votes else None
            gold       = extract_hash_answer(gold_raw)
            ok         = int(
                pred_final is not None
                and gold is not None
                and str(pred_final).strip() == str(gold).strip()
            )
            correct += ok
            records.append({
                "question": q,
                "pred":     pred_final,
                "gold":     gold,
                "correct":  ok,
            })

    acc = correct / len(ds) if ds else 0.0

    save_json(
        {
            "records":         records,
            "gsm8k_maj1":      acc,
            "eval_split":      cfg.gsm8k_eval_split,
            "sampling_k":      cfg.sampling_k,
            "eval_batch_size": eval_bs,
        },
        paths["evals"] / f"gsm8k_{cfg.gsm8k_eval_split}_eval.json",
    )

    _ensure_right_padding(tokenizer)
    return {"gsm8k_maj1": acc}


def run_selected_evals(model, tokenizer, cfg: RunConfig, paths) -> Dict[str, Any]:
    if cfg.eval_buckets is None or not cfg.eval_buckets:
        cfg.eval_buckets = default_eval_buckets(cfg.train_track, cfg.mode)
    print(f"[INFO] Eval buckets: {cfg.eval_buckets}")
    metrics = {}
    if "gsm8k" in cfg.eval_buckets:
        try:
            metrics.update(evaluate_gsm8k(model, tokenizer, cfg, paths))
        except Exception as e:
            print("[WARN] GSM8K eval failed:", e)
    return metrics

print("Evaluation functions (batched GSM8K) defined.")

Evaluation functions (batched GSM8K) defined.


## Cell 16 — Training Loop (FIXED: step ordering + scheduler continuity)

In [78]:
def train_model(
    cfg: RunConfig,
    train_loader,
    tokenizer,
    paths,
    run_id,
    train_tokenized=None,
    collator=None,
):
    start_time  = time.time()
    total_steps = math.ceil(len(train_loader) / cfg.grad_accum_steps) * cfg.epochs

    loraga_loader = None
    if cfg.method == "loraga":
        if train_tokenized is None or collator is None:
            raise ValueError("LoRA-GA requires train_tokenized and collator.")
        loraga_loader = build_loraga_grad_loader(train_tokenized, collator, cfg)

    model = build_model_for_method(
        cfg, train_loader_for_loraga=loraga_loader, total_steps=total_steps
    )

    optimizer, scheduler = rebuild_optimizer(model, cfg, total_steps)

    update_interval     = None
    current_layer_modes: Dict[int, str] = {}
    dynamic_history     = []

    # Logging structures — all saved to disk, generate paper figures
    loss_log:      List[Dict] = []
    lr_log:        List[Dict] = []
    grad_norm_log: List[Dict] = []
    param_log:     List[Dict] = []

    if cfg.method == "automode":
        all_layer_ids       = get_all_transformer_layer_ids(model)
        current_layer_modes = {idx: "lora" for idx in all_layer_ids}
        steps_per_epoch     = math.ceil(len(train_loader) / cfg.grad_accum_steps)
        update_interval     = max(1, steps_per_epoch // max(cfg.dynamic_updates, 1))
        print(
            f"[automode] update_interval={update_interval} steps "
            f"(u={cfg.dynamic_updates}, steps_per_epoch={steps_per_epoch})"
        )

    model.train()
    global_step  = 0
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for epoch in range(cfg.epochs):
        epoch_start   = time.time()
        epoch_loss    = 0.0
        epoch_batches = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.epochs}", leave=True)

        for step, batch in enumerate(pbar):

            batch   = {k: v.to(model.device) for k, v in batch.items()}
            outputs = model(**batch)
            loss    = outputs.loss / cfg.grad_accum_steps
            loss.backward()
            running_loss  += loss.item()
            epoch_loss    += loss.item()
            epoch_batches += 1

            if (step + 1) % cfg.grad_accum_steps == 0:

                should_switch = (
                    cfg.method == "automode"
                    and update_interval is not None
                    and (global_step + 1) % update_interval == 0
                )

                scores = {}
                if should_switch:
                    scores = collect_layer_grad_scores(model)

                # Grad norm BEFORE clip
                total_norm = 0.0
                for p in model.parameters():
                    if p.grad is not None:
                        total_norm += p.grad.detach().float().norm(2).item() ** 2
                total_norm = math.sqrt(total_norm)

                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1

                current_lr = scheduler.get_last_lr()[0]

                # Log everything at this step
                loss_log.append({
                    "step":  global_step,
                    "epoch": epoch + 1,
                    "loss":  round(running_loss * cfg.grad_accum_steps, 6),
                })
                lr_log.append({
                    "step": global_step,
                    "lr":   current_lr,
                })
                grad_norm_log.append({
                    "step":      global_step,
                    "grad_norm": round(total_norm, 6),
                })
                trainable_now, _ = count_trainable_params(model)
                param_log.append({
                    "step":             global_step,
                    "epoch":            epoch + 1,
                    "trainable_params": trainable_now,
                })

                if should_switch and scores:
                    vals      = np.array(list(scores.values()))
                    threshold = float(np.percentile(vals, cfg.dynamic_threshold))
                    selected_full_ft = {
                        idx for idx, sc in scores.items() if sc >= threshold
                    }
                    for idx in get_all_transformer_layer_ids(model):
                        new_mode  = "full_ft" if idx in selected_full_ft else "lora"
                        prev_mode = current_layer_modes.get(idx, "lora")
                        if new_mode != prev_mode:
                            set_automode_layer_mode(model, idx, new_mode, prev_mode=prev_mode)
                            current_layer_modes[idx] = new_mode

                    optimizer = rebuild_optimizer_after_switch(model, cfg)
                    optimizer.zero_grad(set_to_none=True)

                    n_fft  = sum(1 for m in current_layer_modes.values() if m == "full_ft")
                    n_lora = sum(1 for m in current_layer_modes.values() if m == "lora")
                    trainable_after, _ = count_trainable_params(model)

                    print(
                        f"[automode] step={global_step} | "
                        f"threshold={threshold:.4e} | "
                        f"FFT={n_fft} | LoRA={n_lora} | "
                        f"trainable={trainable_after:,}"
                    )
                    dynamic_history.append({
                        "step":                    global_step,
                        "epoch":                   epoch + 1,
                        "scores":                  {k: float(v) for k, v in scores.items()},
                        "threshold":               threshold,
                        "selected_full_ft_layers": sorted(selected_full_ft),
                        "layer_modes":             dict(current_layer_modes),
                        "trainable_params":        trainable_after,
                        "n_fft_layers":            n_fft,
                        "n_lora_layers":           n_lora,
                    })

                pbar.set_postfix({
                    "loss": round(running_loss, 4),
                    "lr":   f"{current_lr:.2e}",
                    "step": global_step,
                })
                running_loss = 0.0

        epoch_time = time.time() - epoch_start
        avg_loss   = (epoch_loss * cfg.grad_accum_steps) / max(epoch_batches, 1)
        print(
            f"[epoch {epoch+1}] avg_loss={avg_loss:.4f} | "
            f"time={epoch_time:.1f}s | "
            f"trainable={count_trainable_params(model)[0]:,}"
        )

    train_time_sec = time.time() - start_time
    peak_vram_gb   = (
        torch.cuda.max_memory_allocated() / (1024 ** 3)
        if torch.cuda.is_available() else 0.0
    )

    # Save all logs
    save_json({"loss_log":      loss_log},      paths["logs"] / "training_loss.json")
    save_json({"lr_log":        lr_log},        paths["logs"] / "lr_schedule.json")
    save_json({"grad_norm_log": grad_norm_log}, paths["logs"] / "grad_norms.json")
    save_json({"param_log":     param_log},     paths["logs"] / "trainable_params.json")

    if cfg.method == "automode":
        save_json({"history": dynamic_history}, paths["dynamic"] / "dynamic_layers.json")

    if cfg.save_model:
        save_dir = paths["checkpoints"] / "final_model"
        model.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)
        print(f"[checkpoint] Saved to {save_dir}")

    metrics = run_selected_evals(model, tokenizer, cfg, paths)
    return model, metrics, train_time_sec, peak_vram_gb

print("train_model defined (all fixes applied).")

train_model defined (all fixes applied).


## Cell 17 — Result Row Schema and CSV Writer

In [79]:
def cfg_to_variant(c: RunConfig) -> str:
    if c.method == "automode":  return f"automode_u{c.dynamic_updates}_t{c.dynamic_threshold}_r{c.lora_r}"
    if c.method == "lora":      return f"lora_r{c.lora_r}_a{c.lora_alpha}"
    if c.method == "dora":      return f"dora_r{c.lora_r}_a{c.lora_alpha}"
    if c.method == "loraga":    return f"loraga_r{c.lora_r}_a{c.lora_alpha}_g{c.loraga_steps_for_grad_est}"
    if c.method == "adalora":   return f"adalora_init{c.adalora_init_r}_target{c.adalora_target_r}"
    if c.method == "topk_full":
        return (f"topk_deepblock_{c.deep_block_start}_{c.deep_block_end}"
                if c.topk_strategy == "deep_block"
                else f"topk_{c.topk_strategy}_k{c.topk_k}")
    if c.method == "bitfit":    return "bitfit"
    if c.method == "full_ft":   return "full_ft"
    return c.method

def cfg_and_metrics_to_result_row(
    c: RunConfig, run_id: str, metrics: Dict[str, Any],
    train_time_sec, peak_vram_gb, status: str = "ok",
) -> Dict[str, Any]:
    row = {
        "run_id": run_id, "method": c.method, "variant": cfg_to_variant(c),
        "eval_batch_size": c.eval_batch_size,
        "seed": c.seed, "mode": c.mode, "train_track": c.train_track,
        "eval_buckets": ",".join(c.eval_buckets or []),
        "model": c.model_name, "lr": c.learning_rate, "epochs": c.epochs,
        "batch_size": c.train_batch_size, "grad_accum": c.grad_accum_steps,
        "use_4bit": c.use_4bit, "dtype": c.dtype_str,
        "gradient_checkpointing": c.gradient_checkpointing,
        "max_train_samples": c.max_train_samples,
        "max_eval_samples": c.max_eval_samples,
        "max_source_len": c.max_source_len, "max_target_len": c.max_target_len,
        "max_new_tokens": c.max_new_tokens, "sampling_k": c.sampling_k,
        "gsm8k_eval_split": c.gsm8k_eval_split,
        "gsm8k_train_on_full_train": c.gsm8k_train_on_full_train,
        "train_time_sec": train_time_sec, "peak_vram_gb": peak_vram_gb,
        "status": status,
        "gsm8k_maj1": metrics.get("gsm8k_maj1"),
        "math_accuracy": metrics.get("math_accuracy"),
        "mmlu_5shot_acc": metrics.get("mmlu_5shot_acc"),
        "arc_challenge_acc": metrics.get("arc_challenge_acc"),
        "lora_r": None, "lora_alpha": None, "lora_dropout": None,
        "automode_u": None, "automode_t": None,
        "topk_strategy": None, "topk_k": None,
        "deep_block_start": None, "deep_block_end": None,
        "adalora_init_r": None, "adalora_target_r": None,
        "loraga_steps_for_grad_est": None,
    }
    if c.method in {"lora", "dora", "loraga", "automode", "adalora"}:
        row.update({"lora_r": c.lora_r, "lora_alpha": c.lora_alpha, "lora_dropout": c.lora_dropout})
    if c.method == "automode":
        row.update({"automode_u": c.dynamic_updates, "automode_t": c.dynamic_threshold})
    if c.method == "topk_full":
        row.update({"topk_strategy": c.topk_strategy, "topk_k": c.topk_k,
                    "deep_block_start": c.deep_block_start, "deep_block_end": c.deep_block_end})
    if c.method == "adalora":
        row.update({"adalora_init_r": c.adalora_init_r, "adalora_target_r": c.adalora_target_r})
    if c.method == "loraga":
        row["loraga_steps_for_grad_est"] = c.loraga_steps_for_grad_est
    return row

CANONICAL_RESULT_COLS = [
    "run_id", "method", "variant", "seed", "mode", "train_track", "eval_buckets",
    "model", "lr", "epochs", "batch_size", "grad_accum", "use_4bit", "dtype",
    "gradient_checkpointing", "max_train_samples", "max_eval_samples",
    "max_source_len", "max_target_len", "max_new_tokens", "sampling_k",
    "gsm8k_eval_split", "gsm8k_train_on_full_train",
    "lora_r", "lora_alpha", "lora_dropout",
    "automode_u", "automode_t",
    "topk_strategy", "topk_k", "deep_block_start", "deep_block_end",
    "adalora_init_r", "adalora_target_r",
    "loraga_steps_for_grad_est",
    "train_time_sec", "peak_vram_gb",
    "gsm8k_maj1", "math_accuracy", "mmlu_5shot_acc", "arc_challenge_acc",
    "status",
]

def save_canonical_results_csv(rows: List[Dict[str, Any]], out_csv: str):
    df = pd.DataFrame(rows)
    for col in CANONICAL_RESULT_COLS:
        if col not in df.columns:
            df[col] = None
    df = df[CANONICAL_RESULT_COLS]
    df.to_csv(out_csv, index=False)
    return df

print("Result schema defined.")

Result schema defined.


## Cell 18 — Sanity Check (run ONCE before experiments)

In [80]:
def run_automode_sanity_check():
    """
    Verifies the complete AutoMode init + LoRA→FFT→LoRA round-trip
    before any real training. Catches PEFT version quirks early.
    """
    print("=" * 65)
    print("[sanity] AutoMode init + switch round-trip check")
    print("=" * 65)

    test_cfg         = RunConfig()
    test_cfg.method  = "automode"
    test_cfg.use_4bit = False   # float model for clean merge/unmerge

    m = load_base_model(test_cfg)
    m = initialize_automode(m, test_cfg)

    trainable_init, total = count_trainable_params(m)
    pct = 100 * trainable_init / total
    print(f"\n[sanity] After init (all LoRA):")
    print(f"         trainable={trainable_init:,} / total={total:,} ({pct:.3f}%)")

    assert trainable_init < 15_000_000, (
        f"FAIL: {trainable_init:,} trainable params — lm_head/embed_tokens likely not frozen!"
    )
    print("[sanity] PASS: param count in LoRA-only range.")

    for name, p in m.named_parameters():
        if "lm_head" in name or "embed_tokens" in name:
            assert not p.requires_grad, f"FAIL: {name} is trainable after init!"
    print("[sanity] PASS: lm_head and embed_tokens are frozen.")

    all_layers = get_all_transformer_layer_ids(m)
    test_layer = all_layers[0]

    print(f"\n[sanity] Switching layer {test_layer}: LoRA → FFT ...")
    set_automode_layer_mode(m, test_layer, "full_ft", prev_mode="lora")
    trainable_fft, _ = count_trainable_params(m)
    print(f"[sanity] PASS: trainable after FFT switch = {trainable_fft:,}")

    print(f"[sanity] Switching layer {test_layer}: FFT → LoRA ...")
    set_automode_layer_mode(m, test_layer, "lora", prev_mode="full_ft")
    trainable_back, _ = count_trainable_params(m)
    print(f"[sanity] PASS: trainable after LoRA restore = {trainable_back:,}")

    diff = abs(trainable_back - trainable_init)
    assert diff < 100_000, (
        f"FAIL: param count after round-trip ({trainable_back:,}) "
        f"doesn't match init ({trainable_init:,}). diff={diff:,}"
    )
    print("[sanity] PASS: param count consistent after full round-trip.")
    print("\n[sanity] All checks passed. Safe to run experiments.")
    print("=" * 65)

    del m
    cleanup_memory()

run_automode_sanity_check()

[sanity] AutoMode init + switch round-trip check


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]


[sanity] After init (all LoRA):
         trainable=3,194,880 / total=2,617,536,768 (0.122%)
[sanity] PASS: param count in LoRA-only range.
[sanity] PASS: lm_head and embed_tokens are frozen.

[sanity] Switching layer 0: LoRA → FFT ...
[sanity] PASS: trainable after FFT switch = 3,194,880
[sanity] Switching layer 0: FFT → LoRA ...
[sanity] PASS: trainable after LoRA restore = 3,194,880
[sanity] PASS: param count consistent after full round-trip.

[sanity] All checks passed. Safe to run experiments.


## Cell 19 — Preset Config Builders

In [81]:
def preset_full_ft(track="gsm8k", seed=8):
    return RunConfig(train_track=track, method="full_ft", seed=seed)

def preset_lora(track="gsm8k", r=16, alpha=32, seed=8):
    return RunConfig(train_track=track, method="lora", lora_r=r, lora_alpha=alpha, seed=seed)

def preset_bitfit(track="gsm8k", seed=8):
    return RunConfig(train_track=track, method="bitfit", seed=seed)

def preset_topk(track="gsm8k", k=6, strategy="deep_block", seed=8):
    return RunConfig(train_track=track, method="topk_full", topk_k=k, topk_strategy=strategy, seed=seed)

def preset_automode(track="gsm8k", r=16, alpha=32, u=10, t=10, seed=8):
    c = RunConfig(train_track=track, method="automode", lora_r=r, lora_alpha=alpha, seed=seed)
    c.dynamic_updates   = u
    c.dynamic_threshold = t
    return c

def preset_adalora(track="gsm8k", seed=8):
    return RunConfig(train_track=track, method="adalora", seed=seed)

def preset_dora(track="gsm8k", r=16, alpha=32, seed=8):
    return RunConfig(train_track=track, method="dora", lora_r=r, lora_alpha=alpha, seed=seed)

def preset_loraga(track="gsm8k", r=16, alpha=32, seed=8):
    return RunConfig(train_track=track, method="loraga", lora_r=r, lora_alpha=alpha, seed=seed)

print("Preset builders defined.")

Preset builders defined.


## Cell 20 — Final Test Config Grid

In [82]:
def make_gsm8k_final_test_cfgs():
    cfgs   = []
    common = dict(
        mode                      = "final_test",
        train_track               = "gsm8k",
        eval_buckets              = ["gsm8k"],
        seed                      = 8,
        epochs                    = 2,
        max_train_samples         = None,
        max_eval_samples          = None,
        max_source_len            = 384,
        max_target_len            = 128,
        max_new_tokens            = 256,
        sampling_k                = 3,
        eval_batch_size           = 8,
        use_4bit                  = False,   # ← single place, all methods same precision
        output_root               = FINAL_ROOT,
        save_model                = False,
        gsm8k_eval_split          = "test",
        gsm8k_val_ratio           = 0.1,
        gsm8k_split_seed          = 123,
        gsm8k_train_on_full_train = True,
    )

    def _apply(c):
        for k, v in common.items():
            setattr(c, k, v)
        return c

    c = _apply(preset_lora(r=16, alpha=32))
    c.save_model = True
    cfgs.append(c)

    c = _apply(preset_lora(r=32, alpha=64))
    c.save_model = False
    cfgs.append(c)

    c = _apply(preset_dora(r=16, alpha=32))
    c.save_model = False
    cfgs.append(c)

    c = _apply(preset_adalora())
    c.save_model = False
    cfgs.append(c)

    c = _apply(preset_bitfit())
    c.save_model = False
    cfgs.append(c)

    c = _apply(preset_topk(k=6, strategy="deep_block"))
    c.save_model = False
    cfgs.append(c)

    for u, t in [(6, 10), (6, 25), (10, 10), (10, 25)]:
        c = _apply(preset_automode(r=16, alpha=32, u=u, t=t))
        c.save_model = (u == 6 and t == 10) or (u == 10 and t == 10)
        cfgs.append(c)

    c = _apply(preset_loraga(r=16, alpha=32))
    c.save_model                = False
    c.loraga_steps_for_grad_est = 16
    cfgs.append(c)

    c = _apply(preset_full_ft())
    c.save_model = True
    cfgs.append(c)

    return cfgs

final_cfgs = make_gsm8k_final_test_cfgs()
print(f"Total runs: {len(final_cfgs)}")
for c in final_cfgs:
    save_flag = "SAVING" if c.save_model else "      "
    if c.method == "automode":
        print(f"{save_flag} | {c.method:10s} | u={c.dynamic_updates}, t={c.dynamic_threshold}, r={c.lora_r}, 4bit={c.use_4bit}")
    elif c.method in {"lora", "dora", "loraga"}:
        print(f"{save_flag} | {c.method:10s} | r={c.lora_r}, a={c.lora_alpha}, 4bit={c.use_4bit}")
    elif c.method == "adalora":
        print(f"{save_flag} | {c.method:10s} | init_r={c.adalora_init_r}, target_r={c.adalora_target_r}, 4bit={c.use_4bit}")
    elif c.method == "topk_full":
        print(f"{save_flag} | {c.method:10s} | strategy={c.topk_strategy}, block=({c.deep_block_start},{c.deep_block_end}), 4bit={c.use_4bit}")
    else:
        print(f"{save_flag} | {c.method:10s} | 4bit={c.use_4bit}")

Total runs: 12
SAVING | lora       | r=16, a=32, 4bit=False
       | lora       | r=32, a=64, 4bit=False
       | dora       | r=16, a=32, 4bit=False
       | adalora    | init_r=16, target_r=8, 4bit=False
       | bitfit     | 4bit=False
       | topk_full  | strategy=deep_block, block=(13,19), 4bit=False
SAVING | automode   | u=6, t=10, r=16, 4bit=False
       | automode   | u=6, t=25, r=16, 4bit=False
SAVING | automode   | u=10, t=10, r=16, 4bit=False
       | automode   | u=10, t=25, r=16, 4bit=False
       | loraga     | r=16, a=32, 4bit=False
SAVING | full_ft    | 4bit=False


## Cell 21 — Run All Experiments (resume-safe)

In [83]:
from IPython.display import display

canonical_csv = str(Path(FINAL_ROOT) / "gsm8k_final_test_canonical.csv")

# Resume-safe: reload completed rows if the CSV already exists
if Path(canonical_csv).exists():
    existing_df    = pd.read_csv(canonical_csv)
    completed_ids  = set(existing_df["run_id"].tolist())
    final_results  = existing_df.to_dict("records")
    print(f"[resume] Found {len(completed_ids)} completed runs in {canonical_csv}")
else:
    completed_ids = set()
    final_results = []

for cfg in final_cfgs:
    run_id_preview = make_run_id(cfg)
    if run_id_preview in completed_ids:
        print(f"[skip] {cfg.method} {cfg_to_variant(cfg)} already completed (run_id={run_id_preview})")
        continue

    print("\n" + "=" * 90)
    print(f"FINAL GSM8K TEST | method={cfg.method} | seed={cfg.seed} | variant={cfg_to_variant(cfg)}")
    if cfg.method == "automode":
        print(f"AutoMode u={cfg.dynamic_updates}, t={cfg.dynamic_threshold}")
    print("=" * 90)

    try:
        set_seed(cfg.seed)

        run_id = make_run_id(cfg)
        paths  = ensure_dirs(cfg, run_id)
        save_json(asdict(cfg), paths["configs"] / "run_config.json")

        tokenizer = get_tokenizer(cfg.model_name)

        train_ds         = load_training_track(cfg, tokenizer)
        train_split_name = "train" if "train" in train_ds else list(train_ds.keys())[0]
        train_tokenized  = build_train_dataset(train_ds[train_split_name], tokenizer, cfg)

        collator     = CausalLMCollator(tokenizer)
        train_loader = DataLoader(
            train_tokenized,
            batch_size=cfg.train_batch_size,
            shuffle=True,
            collate_fn=collator,
        )

        model, metrics, train_time_sec, peak_vram_gb = train_model(
            cfg=cfg,
            train_loader=train_loader,
            tokenizer=tokenizer,
            paths=paths,
            run_id=run_id,
            train_tokenized=train_tokenized,
            collator=collator,
        )

        row = cfg_and_metrics_to_result_row(
            c=cfg, run_id=run_id, metrics=metrics,
            train_time_sec=train_time_sec, peak_vram_gb=peak_vram_gb, status="ok",
        )
        completed_ids.add(run_id)

        del model, tokenizer, train_ds, train_tokenized, train_loader, collator
        cleanup_memory()

    except Exception as e:
        import traceback
        print(f"[ERROR] {cfg.method} failed: {repr(e)}")
        traceback.print_exc()
        row = cfg_and_metrics_to_result_row(
            c=cfg, run_id="FAILED", metrics={},
            train_time_sec=None, peak_vram_gb=None, status=f"failed: {repr(e)}",
        )
        cleanup_memory()

    final_results.append(row)

    df = save_canonical_results_csv(final_results, canonical_csv)
    display(df[["method", "variant", "gsm8k_maj1", "train_time_sec", "peak_vram_gb", "status"]]
            .sort_values("gsm8k_maj1", ascending=False, na_position="last"))

print("\n" + "=" * 90)
print("ALL RUNS COMPLETE")
print(f"Results saved to: {canonical_csv}")
print("=" * 90)


FINAL GSM8K TEST | method=lora | seed=8 | variant=lora_r16_a32


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

{'method': 'lora', 'trainable_params': 3194880, 'total_params': 2617536768, 'trainable_pct': 0.1221}


Epoch 1/2: 100%|██████████| 7473/7473 [27:57<00:00,  4.45it/s, loss=0.474, lr=2.58e-05, step=467]


[epoch 1] avg_loss=0.5017 | time=1677.5s | trainable=3,194,880


...

[epoch 2] avg_loss=0.2059 | time=1547.0s | trainable=2,614,341,888


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[checkpoint] Saved to runs_gemma2_2b_gsm8k_final_test/8708670831b0/checkpoints/final_model
[INFO] Eval buckets: ['gsm8k']


,method,variant,gsm8k_maj1,train_time_sec,peak_vram_gb,status
7,automode,automode_u6_t25_r16,0.469295,3362.607651,13.392385,ok
4,bitfit,bitfit,0.456406,2843.270698,11.820747,ok
1,lora,lora_r32_a64,0.451099,3421.166581,11.820747,ok
6,automode,automode_u6_t10_r16,0.450341,3306.871721,13.392385,ok
9,automode,automode_u10_t25_r16,0.449583,3409.614955,13.392385,ok
8,automode,automode_u10_t10_r16,0.447309,3466.791837,13.392385,ok
3,adalora,adalora_init16_target8,0.421531,4019.575532,11.820747,ok
2,dora,dora_r16_a32,0.414708,4240.670208,11.820747,ok
0,lora,lora_r16_a32,0.406368,3336.649013,11.820747,ok
5,topk_full,topk_deepblock_13_19,0.232752,2815.240792,13.392385,ok



ALL RUNS COMPLETE
Results saved to: runs_gemma2_2b_gsm8k_final_test/gsm8k_final_test_canonical.csv


In [85]:
pip install matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 86.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 114.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 72.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [matplotlib]6 [matplotlib]
Note: you may need to restart the kernel to use updated packages.


In [86]:
# ════════════════════════════════════════════════════════════════
# CELL 22 — Post-run analysis: generate all NeurIPS figures
# Run this after ALL experiments complete.
# ════════════════════════════════════════════════════════════════
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

FIGURES_DIR = Path(FINAL_ROOT) / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

# ── Helpers ───────────────────────────────────────────────────────────────────
def load_run_log(run_id: str, log_name: str):
    p = Path(FINAL_ROOT) / run_id / "logs" / log_name
    return json.load(open(p)) if p.exists() else None

def load_dynamic_log(run_id: str):
    p = Path(FINAL_ROOT) / run_id / "dynamic" / "dynamic_layers.json"
    return json.load(open(p)) if p.exists() else None

METHOD_COLORS = {
    "automode": "#1565C0",
    "full_ft":  "#B71C1C",
    "lora":     "#2E7D32",
    "dora":     "#6A1B9A",
    "adalora":  "#E65100",
    "loraga":   "#00695C",
    "topk_full":"#4E342E",
    "bitfit":   "#607D8B",
}

# ── Load master CSV ───────────────────────────────────────────────────────────
canonical_csv = Path(FINAL_ROOT) / "gsm8k_final_test_canonical.csv"
assert canonical_csv.exists(), "No results CSV yet. Run experiments first."
results_df = pd.read_csv(canonical_csv)
results_df = results_df[results_df["status"] == "ok"].copy()
print(f"Loaded {len(results_df)} completed runs.")
display(results_df[["method","variant","gsm8k_maj1","train_time_sec","peak_vram_gb"]]
        .sort_values("gsm8k_maj1", ascending=False))


# ── FIGURE 1: Main results bar chart ─────────────────────────────────────────
def plot_results_bar(df):
    fig, ax = plt.subplots(figsize=(14, 6))
    df = df.sort_values("gsm8k_maj1", ascending=False)
    x      = range(len(df))
    colors = [METHOD_COLORS.get(r["method"], "#888888") for _, r in df.iterrows()]
    bars   = ax.bar(x, df["gsm8k_maj1"] * 100, color=colors, alpha=0.85, width=0.6)

    for bar, (_, row) in zip(bars, df.iterrows()):
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.3,
                f"{h:.1f}%", ha="center", va="bottom", fontsize=8, rotation=45)

    ax.set_xticks(list(x))
    labels = [r["variant"].replace("automode_","AM_").replace("_r16","")
              for _, r in df.iterrows()]
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
    ax.set_ylabel("GSM8K maj@1 (%)", fontsize=12)
    ax.set_title("GSM8K maj@1 — Gemma-2-2B-IT (seed=8)", fontsize=13)
    ax.set_ylim(0, min(100, df["gsm8k_maj1"].max()*100 + 12))
    ax.grid(axis="y", alpha=0.3)

    handles = [mpatches.Patch(color=c, label=m.replace("_"," ").title())
               for m, c in METHOD_COLORS.items() if m in df["method"].values]
    ax.legend(handles=handles, fontsize=8, ncol=2, loc="lower right")
    plt.tight_layout()
    fig.savefig(FIGURES_DIR/"fig1_results_bar.pdf", dpi=150, bbox_inches="tight")
    fig.savefig(FIGURES_DIR/"fig1_results_bar.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: fig1_results_bar")


# ── FIGURE 2: Pareto frontier (accuracy vs training time) ─────────────────────
def plot_pareto(df):
    df = df.dropna(subset=["gsm8k_maj1","train_time_sec"])
    fig, ax = plt.subplots(figsize=(9, 6))

    for _, row in df.iterrows():
        color  = METHOD_COLORS.get(row["method"], "#888888")
        marker = "*" if row["method"] == "automode" else "o"
        size   = 220 if row["method"] == "automode" else 90
        ax.scatter(row["train_time_sec"]/60, row["gsm8k_maj1"]*100,
                   c=color, marker=marker, s=size, alpha=0.9,
                   edgecolors="white", linewidths=0.5, zorder=5)
        label = (row["variant"].replace("automode_","AM_")
                               .replace("_r16","").replace("lora_r16_a32","LoRA-r16"))
        ax.annotate(label, (row["train_time_sec"]/60, row["gsm8k_maj1"]*100),
                    textcoords="offset points", xytext=(5,3), fontsize=7.5)

    ax.set_xlabel("Training time (minutes)", fontsize=12)
    ax.set_ylabel("GSM8K maj@1 (%)", fontsize=12)
    ax.set_title("Accuracy vs training time — Pareto frontier (2B GSM8K)", fontsize=12)
    ax.grid(True, alpha=0.3)
    handles = [mpatches.Patch(color=c, label=m.replace("_"," ").title())
               for m, c in METHOD_COLORS.items() if m in df["method"].values]
    ax.legend(handles=handles, fontsize=8, loc="lower right")
    plt.tight_layout()
    fig.savefig(FIGURES_DIR/"fig2_pareto.pdf", dpi=150, bbox_inches="tight")
    fig.savefig(FIGURES_DIR/"fig2_pareto.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: fig2_pareto")


# ── FIGURE 3: Training loss curves ───────────────────────────────────────────
def plot_loss_curves(df):
    fig, ax = plt.subplots(figsize=(11, 5))
    for _, row in df.iterrows():
        log = load_run_log(row["run_id"], "training_loss.json")
        if log is None:
            continue
        steps  = [e["step"] for e in log["loss_log"]]
        losses = pd.Series([e["loss"] for e in log["loss_log"]]).rolling(10, min_periods=1).mean().tolist()
        color  = METHOD_COLORS.get(row["method"], "#888888")
        ls     = "--" if row["method"] == "automode" else "-"
        lw     = 2.0 if row["method"] == "automode" else 1.2
        label  = row["variant"].replace("automode_","AM_").replace("_r16","")
        ax.plot(steps, losses, label=label, color=color, linestyle=ls,
                linewidth=lw, alpha=0.85)

    ax.set_xlabel("Optimizer step", fontsize=12)
    ax.set_ylabel("Training loss (smoothed)", fontsize=12)
    ax.set_title("Training loss curves — Gemma-2-2B-IT on GSM8K", fontsize=13)
    ax.legend(fontsize=8, ncol=2, loc="upper right")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR/"fig3_loss_curves.pdf", dpi=150, bbox_inches="tight")
    fig.savefig(FIGURES_DIR/"fig3_loss_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: fig3_loss_curves")


# ── FIGURE 4: Trainable parameter trajectory ─────────────────────────────────
def plot_param_trajectory(df):
    fig, ax = plt.subplots(figsize=(11, 5))
    palette = ["#1565C0","#1976D2","#1E88E5","#42A5F5"]
    am_rows = df[df["method"] == "automode"]
    for i, (_, row) in enumerate(am_rows.iterrows()):
        log = load_run_log(row["run_id"], "trainable_params.json")
        if log is None:
            continue
        steps  = [e["step"]             for e in log["param_log"]]
        params = [e["trainable_params"] for e in log["param_log"]]
        label  = row["variant"].replace("automode_","AM_").replace("_r16","")
        ax.plot(steps, [p/1e6 for p in params],
                label=label, color=palette[i % len(palette)], linewidth=1.8)

    # Add LoRA flat baseline
    lora_rows = df[df["method"] == "lora"]
    if not lora_rows.empty:
        log = load_run_log(lora_rows.iloc[0]["run_id"], "trainable_params.json")
        if log:
            steps  = [e["step"] for e in log["param_log"]]
            params = [e["trainable_params"] for e in log["param_log"]]
            ax.plot(steps, [p/1e6 for p in params],
                    label="LoRA (baseline)", color="#2E7D32",
                    linewidth=1.5, linestyle="--", alpha=0.7)

    ax.set_xlabel("Optimizer step", fontsize=12)
    ax.set_ylabel("Trainable parameters (M)", fontsize=12)
    ax.set_title("Dynamic pruning behavior — AutoMode trainable params over training", fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR/"fig4_dynamic_pruning.pdf", dpi=150, bbox_inches="tight")
    fig.savefig(FIGURES_DIR/"fig4_dynamic_pruning.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: fig4_dynamic_pruning")


# ── FIGURE 5: Layer importance heatmap ───────────────────────────────────────
def plot_layer_heatmap(df):
    am_rows = df[df["method"] == "automode"]
    if am_rows.empty:
        print("No automode runs yet.")
        return

    n_cols = len(am_rows)
    fig, axes = plt.subplots(1, n_cols, figsize=(4*n_cols, 7), squeeze=False)

    for col, (_, row) in enumerate(am_rows.iterrows()):
        ax   = axes[0][col]
        dlog = load_dynamic_log(row["run_id"])
        if dlog is None or not dlog.get("history"):
            ax.set_title(f"{row['variant']}\n(no data)")
            continue

        history = dlog["history"]
        all_layers = sorted(set(
            int(k) for e in history for k in e.get("layer_modes",{}).keys()
        ))
        n_layers = len(all_layers)
        n_steps  = len(history)

        # Build FFT/LoRA matrix
        matrix = np.zeros((n_layers, n_steps))
        for t, entry in enumerate(history):
            modes = entry.get("layer_modes", {})
            for li, lidx in enumerate(all_layers):
                matrix[li, t] = 1.0 if modes.get(str(lidx),"lora") == "full_ft" else 0.0

        fft_freq = matrix.mean(axis=1)
        colors   = plt.cm.YlOrRd(fft_freq)
        ax.barh(range(n_layers), fft_freq, color=colors)
        ax.set_yticks(range(n_layers))
        ax.set_yticklabels([f"L{i}" for i in all_layers], fontsize=7)
        ax.set_xlabel("Fraction of time in FFT mode", fontsize=9)
        ax.set_xlim(0, 1)
        ax.invert_yaxis()
        label = row["variant"].replace("automode_","").replace("_r16","")
        ax.set_title(label, fontsize=9)

    fig.suptitle("Layer importance — fraction of switches assigned to FFT (2B GSM8K)",
                 fontsize=12, y=1.02)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR/"fig5_layer_heatmap.pdf", dpi=150, bbox_inches="tight")
    fig.savefig(FIGURES_DIR/"fig5_layer_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: fig5_layer_heatmap")


# ── FIGURE 6: Layer activation timeline (Gantt) ──────────────────────────────
def plot_layer_timeline(df):
    am_rows = df[df["method"] == "automode"]
    if am_rows.empty:
        return

    # Plot for the u=6,t=10 config (best from prior results)
    best = am_rows[am_rows["variant"].str.contains("u6_t10")]
    if best.empty:
        best = am_rows

    row    = best.iloc[0]
    dlog   = load_dynamic_log(row["run_id"])
    if dlog is None or not dlog.get("history"):
        print("No dynamic log for timeline.")
        return

    history    = dlog["history"]
    all_layers = sorted(set(
        int(k) for e in history for k in e.get("layer_modes",{}).keys()
    ))

    fig, ax = plt.subplots(figsize=(13, max(5, len(all_layers)*0.28)))
    C_FFT  = "#E53935"
    C_LORA = "#1E88E5"

    for li, layer_idx in enumerate(all_layers):
        prev_step = 0
        prev_mode = "lora"
        for t, entry in enumerate(history):
            modes    = entry.get("layer_modes", {})
            cur_mode = modes.get(str(layer_idx), "lora")
            cur_step = entry["step"]
            if cur_mode != prev_mode or t == len(history)-1:
                dur   = max(cur_step - prev_step, 1)
                color = C_FFT if prev_mode == "full_ft" else C_LORA
                ax.barh(li, dur, left=prev_step, height=0.7, color=color, alpha=0.85)
                prev_step = cur_step
                prev_mode = cur_mode

    ax.set_yticks(range(len(all_layers)))
    ax.set_yticklabels([f"Layer {i}" for i in all_layers], fontsize=8)
    ax.set_xlabel("Optimizer step", fontsize=11)
    ax.set_title(
        f"Layer activation timeline — {row['variant'].replace('automode_','AM_').replace('_r16','')}",
        fontsize=12
    )
    ax.invert_yaxis()
    ax.legend(handles=[
        mpatches.Patch(color=C_FFT,  label="Full Fine-Tuning"),
        mpatches.Patch(color=C_LORA, label="LoRA-only"),
    ], loc="upper right", fontsize=9)
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR/"fig6_layer_timeline.pdf", dpi=150, bbox_inches="tight")
    fig.savefig(FIGURES_DIR/"fig6_layer_timeline.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: fig6_layer_timeline")


# ── FIGURE 7: LR schedule verification ───────────────────────────────────────
def plot_lr_schedule(df):
    fig, ax = plt.subplots(figsize=(10, 4))
    am_rows = df[df["method"] == "automode"]
    for _, row in am_rows.iterrows():
        log = load_run_log(row["run_id"], "lr_schedule.json")
        if log is None:
            continue
        steps = [e["step"] for e in log["lr_log"]]
        lrs   = [e["lr"]   for e in log["lr_log"]]
        label = row["variant"].replace("automode_","AM_").replace("_r16","")
        ax.plot(steps, lrs, label=label, linewidth=1.5)

    lora_rows = df[df["method"] == "lora"]
    if not lora_rows.empty:
        log = load_run_log(lora_rows.iloc[0]["run_id"], "lr_schedule.json")
        if log:
            steps = [e["step"] for e in log["lr_log"]]
            lrs   = [e["lr"]   for e in log["lr_log"]]
            ax.plot(steps, lrs, label="LoRA (reference)", color="#2E7D32",
                    linewidth=1.5, linestyle="--")

    ax.set_xlabel("Optimizer step", fontsize=11)
    ax.set_ylabel("Learning rate", fontsize=11)
    ax.set_title("LR schedule — single linear decay, no restarts on mode switch", fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR/"fig7_lr_schedule.pdf", dpi=150, bbox_inches="tight")
    fig.savefig(FIGURES_DIR/"fig7_lr_schedule.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: fig7_lr_schedule")


# ── RUN ALL ───────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("Generating all NeurIPS figures...")
print("="*60 + "\n")

plot_results_bar(results_df)
plot_pareto(results_df)
plot_loss_curves(results_df)
plot_param_trajectory(results_df)
plot_layer_heatmap(results_df)
plot_layer_timeline(results_df)
plot_lr_schedule(results_df)

print(f"\nAll figures saved to: {FIGURES_DIR}")
print("Each figure saved as .pdf (paper) and .png (preview)")

Loaded 12 completed runs.


,method,variant,gsm8k_maj1,train_time_sec,peak_vram_gb
7,automode,automode_u6_t25_r16,0.469295,3362.607651,13.392385
4,bitfit,bitfit,0.456406,2843.270698,11.820747
1,lora,lora_r32_a64,0.451099,3421.166581,11.820747
6,automode,automode_u6_t10_r16,0.450341,3306.871721,13.392385
9,automode,automode_u10_t25_r16,0.449583,3409.614955,13.392385
8,automode,automode_u10_t10_r16,0.447309,3466.791837,13.392385
3,adalora,adalora_init16_target8,0.421531,4019.575532,11.820747
2,dora,dora_r16_a32,0.414708,4240.670208,11.820747
0,lora,lora_r16_a32,0.406368,3336.649013,11.820747
5,topk_full,topk_deepblock_13_19,0.232752,2815.240792,13.392385



Generating all NeurIPS figures...

Saved: fig1_results_bar
Saved: fig2_pareto
Saved: fig3_loss_curves
Saved: fig4_dynamic_pruning
Saved: fig5_layer_heatmap
Saved: fig6_layer_timeline
Saved: fig7_lr_schedule

All figures saved to: runs_gemma2_2b_gsm8k_final_test/figures
Each figure saved as .pdf (paper) and .png (preview)
